In [18]:
from dotenv import load_dotenv

"""
Reads a file named .env in your project folder and loads each KEY=value line into the environment variables of the running program. 
Your .env file should contain:
ANTHROPIC_API_KEY=your-key-here
"""
load_dotenv()

# Imports the Anthropic class from the official SDK (pip install anthropic) ....
from anthropic import Anthropic

"""
Creates the client object you use to call the API. You pass no key because the SDK automatically reads the ANTHROPIC_API_KEY environment 
variable, which is why load_dotenv() had to run before this line. The load_dotenv() call placed above the anthropic import isn't 
required for this to work, since the key is only read when Anthropic() runs. Putting all imports at the top is more conventional.
"""
client = Anthropic()

# FIX: Change from "claude-sonnet-5-0" to "claude-sonnet-5" or "claude-sonnet-5-5"
model = "claude-sonnet-5-5" 

"""
Calls the Messages API, the main endpoint for talking to Claude. It sends your request over the internet and waits for the full response. 
The result is stored in message.
"""
message = client.messages.create(
    # Tells the API which model to use. The name on the left is the parameter, and the model on the right is your variable.
    model=model,
    # A required cap on how long the reply can be. A token is a chunk of text, roughly three-quarters of an English word. 
    # If Claude reaches 1000 tokens, it stops mid-answer. It is a ceiling and not a target, so a short answer uses far fewer tokens.
    max_tokens=1000,

    
    #The conversation so far, as a list of message dictionaries. Each has a role ("user" for you, "assistant" for Claude) and content 
    #(the text). Here there is one user message. For a multi-turn chat, you append earlier messages to this list, because the API has no 
    #memory between calls. You send the whole history each time. The closing ) ends the create( call.
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]
)

In [19]:
# Multi Conversation Flow ....

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system_prompt=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    if system_prompt:
        params["system"] = system_prompt
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return "".join(block.text for block in message.content if block.type == "text")

    

In [20]:
messages = []

add_user_message(messages, "Count from 1 to 10")
chat(messages, stop_sequences=["5", "3", "4"])

# Output: '1, 2, '

# A stop sequence is a piece of text that tells the API: "If you ever generate this exact text, stop immediately and don't include it."
# You asked for "Count from 1 to 10", and Claude began writing its normal answer: 1, 2, 3, 4, 5, ...
# The API generates text piece by piece and checks the growing text against your list ["5", "3", "4"] as it goes.
# After 1, 2, the next piece would be 3, which matches one of your sequences. At that moment generation stops.
# The matched string (3) is not included in the output. Everything before it is kept, including the comma and space that Claude 
# had already written after 2.

'1, 2, '

In [21]:

messages = []

# This prints the json along with the explanation after json.
add_user_message(messages, "Generate a very short event bridge rule as json")
chat(messages)

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(
    messages,
    system_prompt="Respond with raw JSON only. No markdown fences, no explanation."
)
print("Text generated is: " , text)

BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'This model does not support assistant message prefill. The conversation must end with a user message.'}, 'request_id': 'req_011Cfv5tVic6tXjRdCZPLX3t'}